# Stage 6: Model Development & Systematic Comparison

This notebook trains four classifiers on the processed dataset and compares them in a systematic, leakage-free way:

| Model | Role | Feature set |
|---|---|---|
| Logistic Regression | Linear baseline | Scaled |
| Decision Tree | Non-linear, captures interactions | Unscaled |
| Random Forest | Bagging ensemble, lowers variance | Unscaled |
| XGBoost | Gradient boosting for tabular data | Unscaled |

**Key principles**
- Models are compared with **Stratified 5-Fold Cross-Validation** on the training data only.
- Ranking uses **CV Macro F1**, never the test score.
- The **hold-out test set** stays independent and is only used to report final performance.

**Prerequisite:** the files `X_train_final.csv`, `X_train_scaled.csv`, `X_test_final.csv`, `X_test_scaled.csv`, `y_train_final.csv` and `y_test_final.csv` must exist in `data/processed/`.

## 1. Imports

Libraries for data handling (`pandas`), the scikit-learn models, cross-validation, label encoding and metrics, plus `XGBClassifier` from `xgboost`.

In [ ]:
from pathlib import Path

import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from xgboost import XGBClassifier

## 2. Configuration

- `DATA_DIR` points to the folder with the processed CSV files.
- `RANDOM_STATE` is fixed so results are reproducible.

In [ ]:
# ============================================================
# STAGE 6: MODEL DEVELOPMENT & SYSTEMATIC COMPARISON
# ============================================================

DATA_DIR = Path("data/processed")
RANDOM_STATE = 42

## 3. The `main()` function

The whole workflow is defined in one function, and it is kept exactly as written so the code is unchanged. Here is what each section inside it does:

1. **Load processed datasets:** reads the unscaled and scaled train/test features and the target labels.
2. **Encode target labels:** `LabelEncoder` is fitted on the training labels only, then applied to the test labels (XGBoost needs numeric classes).
3. **Validation strategy:** `StratifiedKFold` with 5 splits keeps class proportions similar in each fold. Scoring uses accuracy and macro precision, recall and F1.
4. **Define models:** each entry holds the model, its training features and its test features. Logistic Regression uses the scaled features, the tree-based models use the unscaled ones.
5. **Systematic evaluation:** for each model it runs cross-validation on the training data, refits on the full training set and then evaluates once on the hold-out test set.
6. **Model comparison:** results are ranked by **CV Macro F1** and printed.
7. **Save results:** the table is written to `stage6_model_comparison_results.csv`. Stage 7 loads this file as its baseline.
8. **Technical observations:** prints notes on algorithm choice, validation, metric choice and generalization, useful for the Stage 8 viva.

In [ ]:
def main():

    print("=" * 80)
    print("STAGE 6: MODEL DEVELOPMENT & SYSTEMATIC COMPARISON")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Load processed datasets
    # --------------------------------------------------------

    print("\n[1/5] Loading processed datasets...")

    X_train_final = pd.read_csv(
        DATA_DIR / "X_train_final.csv"
    )

    X_train_scaled = pd.read_csv(
        DATA_DIR / "X_train_scaled.csv"
    )

    X_test_final = pd.read_csv(
        DATA_DIR / "X_test_final.csv"
    )

    X_test_scaled = pd.read_csv(
        DATA_DIR / "X_test_scaled.csv"
    )

    y_train_raw = pd.read_csv(
        DATA_DIR / "y_train_final.csv"
    ).values.ravel()

    y_test_raw = pd.read_csv(
        DATA_DIR / "y_test_final.csv"
    ).values.ravel()

    # --------------------------------------------------------
    # 2. Encode target labels
    # --------------------------------------------------------
    # XGBoost requires numeric class labels for multiclass
    # classification.
    #
    # The LabelEncoder is fitted ONLY on the training labels.
    # The same mapping is then applied to the test labels.
    # --------------------------------------------------------

    label_encoder = LabelEncoder()

    y_train = label_encoder.fit_transform(y_train_raw)
    y_test = label_encoder.transform(y_test_raw)

    target_classes = label_encoder.classes_

    print(f"  Training samples : {len(y_train):,}")
    print(f"  Test samples     : {len(y_test):,}")
    print(f"  Feature count    : {X_train_final.shape[1]:,}")

    print(
        "  Target classes   : "
        f"{dict(enumerate(target_classes))}"
    )

    # --------------------------------------------------------
    # 3. Validation strategy
    # --------------------------------------------------------
    # Stratified K-Fold preserves the approximate class
    # distribution in every fold.
    # --------------------------------------------------------

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    scoring = {
        "accuracy": "accuracy",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro",
        "f1_macro": "f1_macro",
    }

    # --------------------------------------------------------
    # 4. Define models
    # --------------------------------------------------------
    #
    # Logistic Regression:
    # Linear baseline. Uses scaled features.
    #
    # Decision Tree:
    # Captures non-linear relationships and interactions.
    #
    # Random Forest:
    # Bagging ensemble that reduces variance and captures
    # non-linear relationships.
    #
    # XGBoost:
    # Gradient boosting model suitable for structured/tabular
    # classification data.
    # --------------------------------------------------------

    models = {

        "Logistic Regression": (
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
            X_train_scaled,
            X_test_scaled,
        ),

        "Decision Tree": (
            DecisionTreeClassifier(
                random_state=RANDOM_STATE
            ),
            X_train_final,
            X_test_final,
        ),

        "Random Forest": (
            RandomForestClassifier(
                n_estimators=100,
                random_state=RANDOM_STATE,
                n_jobs=2,
            ),
            X_train_final,
            X_test_final,
        ),

        "XGBoost": (
            XGBClassifier(
                n_estimators=100,
                random_state=RANDOM_STATE,
                n_jobs=2,
                eval_metric="mlogloss",
            ),
            X_train_final,
            X_test_final,
        ),
    }

    # --------------------------------------------------------
    # 5. Systematic model evaluation
    # --------------------------------------------------------

    print(
        "\n[2/5] Running 5-Fold Stratified "
        "Cross-Validation..."
    )

    results = []

    for model_name, (
        model,
        train_features,
        test_features,
    ) in models.items():

        print(f"\n  -> Evaluating {model_name}...")

        # ----------------------------------------------------
        # Cross-validation on training data only
        # ----------------------------------------------------

        scores = cross_validate(
            model,
            train_features,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=1,
            return_train_score=False,
        )

        # ----------------------------------------------------
        # Fit on the complete training data
        # ----------------------------------------------------

        model.fit(
            train_features,
            y_train
        )

        # ----------------------------------------------------
        # Final hold-out test evaluation
        # ----------------------------------------------------

        y_pred = model.predict(test_features)

        test_acc = accuracy_score(
            y_test,
            y_pred
        )

        test_prec = precision_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0,
        )

        test_rec = recall_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0,
        )

        test_f1 = f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0,
        )

        # ----------------------------------------------------
        # Store results
        # ----------------------------------------------------

        results.append(
            {
                "Model": model_name,

                "CV Accuracy":
                    scores["test_accuracy"].mean(),

                "CV Accuracy Std":
                    scores["test_accuracy"].std(),

                "CV Precision (Macro)":
                    scores["test_precision_macro"].mean(),

                "CV Precision Std":
                    scores["test_precision_macro"].std(),

                "CV Recall (Macro)":
                    scores["test_recall_macro"].mean(),

                "CV Recall Std":
                    scores["test_recall_macro"].std(),

                "CV F1 (Macro)":
                    scores["test_f1_macro"].mean(),

                "CV F1 Std":
                    scores["test_f1_macro"].std(),

                "Test Accuracy":
                    test_acc,

                "Test Precision (Macro)":
                    test_prec,

                "Test Recall (Macro)":
                    test_rec,

                "Test F1 (Macro)":
                    test_f1,
            }
        )

    # --------------------------------------------------------
    # 6. Model comparison
    # --------------------------------------------------------
    #
    # IMPORTANT:
    # Ranking is based on CV Macro F1, NOT test F1.
    #
    # The test set remains an independent hold-out set.
    # --------------------------------------------------------

    summary = (
        pd.DataFrame(results)
        .sort_values(
            "CV F1 (Macro)",
            ascending=False
        )
        .reset_index(drop=True)
    )

    print(
        "\n[3/5] Systematic Model Comparison "
        "(Ranked by CV Macro F1):"
    )

    display_columns = [
        "Model",
        "CV F1 (Macro)",
        "CV F1 Std",
        "CV Precision (Macro)",
        "CV Recall (Macro)",
        "Test F1 (Macro)",
        "Test Accuracy",
    ]

    print(
        summary[display_columns].to_string(
            index=False,
            float_format=lambda value: f"{value:.4f}",
        )
    )

    # --------------------------------------------------------
    # 7. Save experiment results
    # --------------------------------------------------------

    output_file = (
        "stage6_model_comparison_results.csv"
    )

    summary.to_csv(
        output_file,
        index=False
    )

    print(
        f"\n[4/5] Results saved to: {output_file}"
    )

    # --------------------------------------------------------
    # 8. Technical observations
    # --------------------------------------------------------

    print(
        "\n[5/5] Technical Observations "
        "& Stage 8 Viva Guidance:"
    )

    print("-" * 80)

    print(
        "\n1. Algorithm Selection:"
    )

    print(
        "   - Logistic Regression provides a simple linear baseline."
    )

    print(
        "   - Decision Tree captures non-linear relationships "
        "and feature interactions."
    )

    print(
        "   - Random Forest combines multiple decision trees "
        "to reduce variance."
    )

    print(
        "   - XGBoost uses gradient boosting to learn complex "
        "patterns from tabular data."
    )

    print(
        "\n2. Validation Strategy:"
    )

    print(
        "   - Stratified 5-Fold Cross-Validation was used to "
        "preserve class proportions across folds."
    )

    print(
        "\n3. Metric Selection:"
    )

    print(
        "   - Macro F1 is used as the primary comparison metric "
        "because the target classes are imbalanced."
    )

    print(
        "   - Macro F1 gives equal importance to each class and "
        "balances precision and recall."
    )

    print(
        "\n4. Model Comparison:"
    )

    best_model = summary.iloc[0]["Model"]
    best_cv_f1 = summary.iloc[0]["CV F1 (Macro)"]

    print(
        f"   - Highest CV Macro F1 in this experiment: "
        f"{best_model} ({best_cv_f1:.4f})"
    )

    print(
        "\n5. Generalization:"
    )

    print(
        "   - CV and hold-out test performance should be "
        "compared to assess consistency."
    )

    print(
        "   - A large difference between CV and test performance "
        "should be investigated rather than automatically "
        "interpreted as proof of leakage or overfitting."
    )

    print("\n" + "=" * 80)
    print("STAGE 6 COMPLETED")
    print("=" * 80)

## 4. Run Stage 6

Executes the pipeline. Running it also creates `stage6_model_comparison_results.csv` in the working directory, which **Stage 7 requires**.

> Tip: cross-validating four models can take a while, especially Random Forest and XGBoost.

In [ ]:
if __name__ == "__main__":
    main()

## Wrap-up

- Check the ranking table for the model with the highest **CV Macro F1**.
- Compare each model's CV score with its test score. A big gap should be investigated, not automatically assumed to be leakage or overfitting.
- Continue to **Stage 7: Model Optimization & Final Model Selection**.